# 4. Derivatives: conventional and composite

Two readings of "the derivative of this expression", and they are not always the
same number.

* the **conventional** derivative is of the function you had in mind
* the **composite** derivative is of the function the expression *denotes*

Where no zero enters they agree exactly. Where one does, they differ — and the
composite answer is a true derivative of a real function, just not the one you
were thinking of. This notebook shows which function, and how to ask for the other
reading when you want it.

In [1]:
import math, warnings
import composite.composite_lib as cl
from composite import R, ZERO, Composite, sin, cos, exp, ln, sqrt, TAG, UNTAG
from composite.backends.config import use_dict
from mpmath import mp, mpf

use_dict(); cl._refresh_constants()
# MAX_ACTIVE_DIMS is left at its default of 60. Raising it removes the
# truncation guard, and an iterative algorithm then grows terms without
# bound -- solve_ode below did not finish 20 steps in 30s at 10**9, and
# finishes 200 steps in 0.86s at the default.
mp.dps = 40
warnings.simplefilter("ignore")
print("ready")

ready


## 1. Where they agree: everything without a zero

Seed the point, evaluate once, read the grades. Checked against 40-digit
arithmetic, order by order.

In [2]:
pairs = [("sin(x)*exp(-x)",        lambda x: sin(x) * exp(-x),
                                   lambda t: mp.sin(t) * mp.e ** (-t),            0.7),
         ("x**3/(1+x**2)",         lambda x: x ** 3 / (R(1) + x * x),
                                   lambda t: t ** 3 / (1 + t * t),                1.3),
         ("ln(1+x)*cos(x)",        lambda x: ln(R(1) + x) * cos(x),
                                   lambda t: mp.log(1 + t) * mp.cos(t),           0.4)]
for label, f, exact, at in pairs:
    jet = f(R(at) + ZERO)
    worst = 0.0
    for k in range(5):
        want = float(mp.diff(exact, mpf(at), k))
        worst = max(worst, abs(jet.d(k) - want) / max(abs(want), 1.0))
    print("%-20s at %.1f   orders 0..4 agree to %.1e" % (label, at, worst))

sin(x)*exp(-x)       at 0.7   orders 0..4 agree to 1.4e-17
x**3/(1+x**2)        at 1.3   orders 0..4 agree to 6.7e-16
ln(1+x)*cos(x)       at 0.4   orders 0..4 agree to 3.3e-16


No step size, no difference quotient, every order from one evaluation. For a
zero-free expression there is nothing to choose between the two readings.

## 2. The first divergence: a written zero

`x**2` at 3 has derivative 6. Add a zero and the composite answer becomes 7.

In [3]:
x = R(3) + ZERO
plain      = x * x
with_zero  = x * x + R(0)
print("x**2          ", plain,     " d(1) =", plain.d(1))
print("x**2 + R(0)   ", with_zero, " d(1) =", with_zero.d(1))

x**2           <|9|₀ |6|₋₁ |1|₋₂>  d(1) = 6.0
x**2 + R(0)    <|9|₀ |7|₋₁ |1|₋₂>  d(1) = 7.0


That is not a rounding error and not a bug. **A written zero is an infinitesimal**,
because the system has no additive identity. So `x**2 + 0` is not `x**2`; the zero
sits at grade 0, and an expressed zero of magnitude `m` at grade `-k` **denotes**
`m*(x-a)**k`.

With `m = 1` and `k = 0` that is `(x-a)**0`... but the grade-0 zero converts on use
to grade `-1`, which denotes `(x-3)`. So the expression denotes

    x**2 + (x - 3)

whose derivative at 3 is `2*3 + 1 = 7`. The composite is right about the function it
was given.

In [4]:
print("the function the expression denotes, computed independently:")
denoted = x * x + (x - R(3))
print("  x**2 + (x-3)  ->", denoted, " d(1) =", denoted.d(1))
print()
print("same jet as x**2 + R(0):", [with_zero.d(k) for k in range(3)],
      "vs", [denoted.d(k) for k in range(3)])

the function the expression denotes, computed independently:
  x**2 + (x-3)  -> <|9|₀ |7|₋₁ |1|₋₂>  d(1) = 7.0

same jet as x**2 + R(0): [9.0, 7.0, 2.0] vs [9.0, 7.0, 2.0]


## 3. The verification that settles it

A harder case. `x**2 * (2-2) + x**2` looks like it should be `x**2`, since `2-2`
is zero. The composite gives the jet `[9, 24, 26, 12]` at x=3.

Those are not corrupted numbers. They are the exact value, slope, curvature and
third derivative of **`2x**3 - 5x**2`**, computed independently below.

In [5]:
messy  = x * x * (R(2) - R(2)) + x * x
cubic  = R(2) * x ** 3 - R(5) * x * x

print("x**2*(2-2) + x**2  ->", [messy.d(k) for k in range(4)])
print("2x**3 - 5x**2      ->", [cubic.d(k) for k in range(4)])
print()
print("check by hand at x = 3:")
print("  f  = 2*27 - 5*9   =", 2 * 27 - 5 * 9)
print("  f' = 6*9 - 10*3   =", 6 * 9 - 10 * 3)
print("  f''= 12*3 - 10    =", 12 * 3 - 10)
print("  f'''= 12          =", 12)

x**2*(2-2) + x**2  -> [9.0, 24.0, 26.0, 12.0]
2x**3 - 5x**2      -> [9.0, 24.0, 26.0, 12.0]

check by hand at x = 3:
  f  = 2*27 - 5*9   = 9
  f' = 6*9 - 10*3   = 24
  f''= 12*3 - 10    = 26
  f'''= 12          = 12


A real slope and a real acceleration — of that function. The cancellation did not
corrupt the jet; it changed which function the expression denotes.

**The standard part is never affected.** Grade 0 stays 9 in every version. It is
orders 1 and up that move.

## 4. The marker: `denotation_order`

The number records the order from which its jet stops being the classical one, so
you never have to guess whether a reading is denoted.

In [6]:
for label, v in (("x**2",                plain),
                ("x**2 + R(0)",         with_zero),
                ("x**2*(2-2) + x**2",   messy)):
    print("%-22s denotation_order = %s" % (label, v.denotation_order))

x**2                   denotation_order = None
x**2 + R(0)            denotation_order = 1.0
x**2*(2-2) + x**2      denotation_order = 1.0


In [7]:
messy     # the grade table flags every affected order

grade,coefficient,meaning
0,9,the value
-1,24,"d(1) = 24; DENOTED, not the classical derivative"
-2,13,"d(2) = 26; DENOTED, not the classical derivative"
-3,2,"d(3) = 12; DENOTED, not the classical derivative"


## 5. And it tells you, however the zero is spelled

Every spelling that changes the jet records `denotation_order` and warns. The
warning is registered to fire *always*, not once, because the hazard is precisely
that grade 0 stays right while every derivative moves.

In [8]:
builders = [("x*x",                   lambda x: x * x),
            ("x*x + R(0)",            lambda x: x * x + R(0)),
            ("x*x + 0",               lambda x: x * x + 0),
            ("x*x*(R(2)-R(2)) + x*x", lambda x: x * x * (R(2) - R(2)) + x * x)]

print("%-26s %-8s %-8s %s" % ("expression", "d(1)", "denot", "warnings raised"))
for label, build in builders:
    warnings.resetwarnings()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        xx = R(3) + ZERO
        v = cl._ensure_composite(build(xx))      # BUILD inside the block, or you
        d1 = v.d(1)                              # miss construction-time warnings
        names = sorted({c.category.__name__ for c in caught})
    print("%-26s %-8s %-8s %s" % (label, d1, v.denotation_order, names or "NONE"))
warnings.simplefilter("ignore")

expression                 d(1)     denot    warnings raised
x*x                        6.0      None     NONE
x*x + R(0)                 7.0      1.0      ['NotConventionalWarning', 'UserWarning']
x*x + 0                    7.0      1.0      ['NotConventionalWarning', 'UserWarning']
x*x*(R(2)-R(2)) + x*x      24.0     1.0      ['CancellationWarning', 'NotConventionalWarning']


All three disclosing spellings agree, and that took a fix. `R(0)` used to return
`|1|₋₁` **already converted**, which skipped `_r1` — and `_r1` is where the warning
fires and where `denotation_order` is recorded. So `x*x + R(0)` silently read 7
instead of 6 with `denotation_order is None`, while `x*x + 0` with identical
semantics warned twice. The explicit spelling was the undisclosed one.

`R(0)` now returns the latent `|0|₀` and converts at the point of use like any other
written zero. `ZERO` is deliberately **not** marked: it names the infinitesimal
rather than being a written zero, so an ordinary seed `R(a) + ZERO` reports no
denotation.

Still: `denotation_order` tells you a zero entered, not that your reading is wrong.
It is the exact jet of the function the expression denotes — read §3 again if that
sounds like a defect.

## 6. Three ways to ask for the conventional reading

| | when to use it |
|---|---|
| `Composite({})` | you meant *nothing*, not a zero — annihilate with no record |
| `conventional()` | a block of ported code that assumes an additive identity |
| `TAG(a)` / `UNTAG()` | one value carries the infinitesimal; every other zero is ordinary |

In [9]:
print("the problem:")
print("  x**2 + R(0)                  d(1) =", (x * x + R(0)).d(1))
print()
print("NOTHING adds no order:")
print("  x**2 + Composite({})         d(1) =", (x * x + Composite({})).d(1))
print()
print("conventional() for a block:")
with cl.conventional():
    xc = R(3) + ZERO
    print("  x**2 + R(0)                  d(1) =", (xc * xc + R(0)).d(1))
print()
print("TAG, and every other zero goes ordinary:")
xt = TAG(3)
print("  x**2 + R(0)                  d(1) =", (xt * xt + R(0)).d(1))
print("  x**2*(2-2) + x**2            d(1) =", (xt * xt * (R(2) - R(2)) + xt * xt).d(1))
UNTAG()

the problem:
  x**2 + R(0)                  d(1) = 7.0

NOTHING adds no order:
  x**2 + Composite({})         d(1) = 6.0

conventional() for a block:
  x**2 + R(0)                  d(1) = 6.0

TAG, and every other zero goes ordinary:
  x**2 + R(0)                  d(1) = 6.0
  x**2*(2-2) + x**2            d(1) = 6.0


True

## 7. `d(n)` reads; `D(n)` differentiates

`d(n)` reads grade `-n` and multiplies by `n!`, which is the n-th derivative **only**
when the number is a Taylor series in `h`. Put the grades above 0 — a pole — and
grade `-n` is simply absent, so `d(n)` would answer 0 for a derivative that is
unbounded. It raises instead. `D(n)` differentiates and keeps the grade.

In [10]:
pole = R(1) / ZERO                 # 1/h, a first-order pole
print("the number        ", pole)
try:
    pole.d(1)
except Exception as problem:
    print("d(1)              ", type(problem).__name__, "-- correct, the derivative is unbounded")
print("D(1)              ", pole.D(1), " = -1/h**2, still a composite")
print("D(2)              ", pole.D(2))
print()
print("and the half-grade case:")
corner = sqrt(ZERO)
print("sqrt(h)           ", corner)
try:
    corner.d(1)
except Exception as problem:
    print("d(1)              ", type(problem).__name__, "-- a branch point has no finite slope")
print("D(1)              ", corner.D(1))

the number         <|1|₁>
d(1)               StandardPartUndefinedError -- correct, the derivative is unbounded
D(1)               <|-1|₂>  = -1/h**2, still a composite
D(2)               <|2|₃>

and the half-grade case:
sqrt(h)            <1_-0.5>
d(1)               StandardPartUndefinedError -- a branch point has no finite slope
D(1)               <0.5_0.5>


## 8. Where the `n!` comes from

`d(n)` is `coefficient * n!`, because grade `-n` holds the Taylor *coefficient*, not
the derivative. Applying the grade shift `n` times produces the same factorial, so
the two agree on any genuine series.

In [11]:
jet = sin(R(2) + ZERO) * (R(2) + ZERO)
print("%-6s %-24s %-24s %s" % ("n", "coefficient * n!", "D(n) then st()", "agree"))
for n in range(1, 5):
    by_reading = jet.coeffs_dict().get(-n, 0.0) * math.factorial(n)
    by_shifting = jet.D(n).st()
    print("%-6d %-24.15g %-24.15g %s"
          % (n, by_reading, by_shifting, abs(by_reading - by_shifting) < 1e-12))

n      coefficient * n!         D(n) then st()           agree
1      0.0770037537313969       0.0770037537313969       True
2      -2.65088852674565        -2.65088852674565        True
3      -1.89559860738276        -1.89559860738276        True
4      3.48318219983993         3.48318219983993         True


## Summary

| | conventional | composite |
|---|---|---|
| zero-free expression | same | same, to the last bit |
| `x**2 + 0` at 3 | 6 | 7 — of `x**2 + (x-3)` |
| what moves | — | orders 1 and up; grade 0 never |
| told about it? | no | `denotation_order` and a warning on every read |
| derivative of a pole | undefined | `D(n)` gives it; `d(n)` refuses |
| cost of all orders | one evaluation per order | one evaluation |

The rule in one line: **a composite derivative is always a true derivative — of the
function the expression denotes.** If a zero entered, check which function that is.

Next: **5. Integration** — the same grade shift, in the other direction.